In [10]:
import json
import sys
import requests
import os
from dotenv import load_dotenv; load_dotenv()

from IPython.display import display
import pandas as pd

pd.set_option("display.max_columns", None)

API_URL = "https://api.jquants.com"

In [11]:
REFRESH_TOKEN = os.environ.get("REFRESH_TOKEN")

In [13]:
res = requests.post(f"{API_URL}/v1/token/auth_refresh?refreshtoken={REFRESH_TOKEN}")
if res.status_code == 200:
    id_token = res.json()['idToken']
    headers = {'Authorization': 'Bearer {}'.format(id_token)}
    display("idTokenの取得に成功しました。")
else:
    display(res.json()["message"])

'idTokenの取得に成功しました。'

In [16]:
code = "7082"
date = ""

In [17]:
params = {}
if code != "":
  params["code"] = code
if date != "":
  params["date"] = date

res = requests.get(f"{API_URL}/v1/fins/announcement", params=params, headers=headers)

if res.status_code == 200:
  d = res.json()
  data = d["announcement"]
  while "pagination_key" in d:
    params["pagination_key"] = d["pagination_key"]
    res = requests.get(f"{API_URL}/v1/fins/announcement", params=params, headers=headers)
    d = res.json()
    data += d["announcement"]
  df = pd.DataFrame(data)
  display(df)
else:
  print(res.json())

,Date,Code,CompanyName,FiscalYear,SectorName,FiscalQuarter,Section
0,2025-11-19,87250,ＭＳ＆ＡＤインシュアランスグループホールディングス,3月31日,保険業,第２四半期,プライム
1,2025-11-19,87660,東京海上ホールディングス,3月31日,保険業,第２四半期,プライム
2,2025-11-19,86300,ＳＯＭＰＯホールディングス,3月31日,保険業,第２四半期,プライム


In [2]:
import os
import datetime as dt
import requests
import gspread
from google.auth import default
from googleapiclient.discovery import build
from google.oauth2 import service_account
from oauth2client.service_account import ServiceAccountCredentials
from dotenv import load_dotenv

load_dotenv()

# ==== 設定 ====
API_URL = "https://api.jquants.com"
REFRESH_TOKEN = os.environ["REFRESH_TOKEN"]  # .env に入れる
SPREADSHEET_ID = "1WlamXyzIj6GZAkU_lc8C0mTvMzwoHZk-R_HodUC3Sws"   

# 1. idToken 取得
def get_headers():
    res = requests.post(f"{API_URL}/v1/token/auth_refresh", params={"refreshtoken": REFRESH_TOKEN})
    res.raise_for_status()
    id_token = res.json()["idToken"]
    return {"Authorization": f"Bearer {id_token}"}

# 2. 決算予定日をまとめて取得（/fins/announcement は全件→コードごとに次の予定日を算出）
def fetch_next_announcement_map(headers):
    # 直近＆今後の予定日だけを辞書化 code -> YYYY-MM-DD
    params = {}
    data = []
    while True:
        r = requests.get(f"{API_URL}/v1/fins/announcement", params=params, headers=headers)
        r.raise_for_status()
        d = r.json()
        data.extend(d.get("announcement", []))
        if "pagination_key" not in d:
            break
        params["pagination_key"] = d["pagination_key"]
    today = dt.date.today().isoformat()
    next_map = {}
    for row in data:
        code = row.get("code")
        ann_date = row.get("announcement_date") or row.get("date")
        if not code or not ann_date:
            continue
        if ann_date < today:
            continue
        if code not in next_map or ann_date < next_map[code]:
            next_map[code] = ann_date
    return next_map

# 3. 指定コードの最新終値を取得
def fetch_latest_close(headers, code, lookback_days=30):
    today = dt.date.today()
    since = (today - dt.timedelta(days=lookback_days)).isoformat()
    params = {"code": code, "from": since, "to": today.isoformat()}
    latest = None

    while True:
        r = requests.get(f"{API_URL}/v1/prices/daily_quotes", params=params, headers=headers)
        if r.status_code != 200:
            print(f"skip price for {code}: {r.status_code} {r.text}")
            return None
        d = r.json()
        for row in d.get("daily_quotes", []):
            date = row.get("date")
            # 調整済み/前どちらかが入っているので、存在するものを優先的に使う
            close = (
                row.get("close")
                or row.get("adjustment_close")
                or row.get("end_price")  # 念のため
            )
            if date and close is not None:
                if latest is None or date > latest["date"]:
                    latest = {"date": date, "close": close}
        if "pagination_key" not in d:
            break
        params["pagination_key"] = d["pagination_key"]

    return None if latest is None else latest["close"]

# 4. スプレッドシート更新
def update_sheet():
    headers = get_headers()
    scope = [
        "https://spreadsheets.google.com/feeds",
        "https://www.googleapis.com/auth/drive",
    ]
    credentials = ServiceAccountCredentials.from_json_keyfile_name(
        "abiding-ascent-476815-q6-56a05b29f113.json", scope
    )
    client = gspread.authorize(credentials)
    ws = client.open_by_key(SPREADSHEET_ID).worksheet("list")
    # ヘッダー行から列位置を特定
    header_row = ws.row_values(1)
    def col_idx(label):
        return header_row.index(label) + 1
    code_col = col_idx("銘柄コード")
    kessan_col = col_idx("決算予定日")
    price_col = col_idx("現在株価")

    codes = ws.col_values(code_col)[1:]  # 2行目以降
    announcement_map = fetch_next_announcement_map(headers)

    updates = []
    for i, code in enumerate(codes, start=2):  # 行番号は2から
        if not code:
            continue
        next_ann = announcement_map.get(code, "")
        latest_close = fetch_latest_close(headers, code)
        if next_ann:
            updates.append(gspread.Cell(i, kessan_col, next_ann))
        if latest_close is not None:
            updates.append(gspread.Cell(i, price_col, latest_close))

    if updates:
        ws.update_cells(updates, value_input_option="USER_ENTERED")
    print(f"更新セル数: {len(updates)}")

if __name__ == "__main__":
    update_sheet()


skip price for 7082: 400 {"message": "Your subscription covers the following dates: 2023-08-29 ~ 2025-08-29. If you want more data, please check other plans:https://jpx-jquants.com/"}
skip price for 1407: 400 {"message": "Your subscription covers the following dates: 2023-08-29 ~ 2025-08-29. If you want more data, please check other plans:https://jpx-jquants.com/"}
skip price for 6224: 400 {"message": "Your subscription covers the following dates: 2023-08-29 ~ 2025-08-29. If you want more data, please check other plans:https://jpx-jquants.com/"}
skip price for 5139: 400 {"message": "Your subscription covers the following dates: 2023-08-29 ~ 2025-08-29. If you want more data, please check other plans:https://jpx-jquants.com/"}
skip price for 4398: 400 {"message": "Your subscription covers the following dates: 2023-08-29 ~ 2025-08-29. If you want more data, please check other plans:https://jpx-jquants.com/"}
skip price for 7760: 400 {"message": "Your subscription covers the following dat